In [ ]:
import pandas as pd
import re, json, os
from collections import Counter
import numpy as np

# ── Paths ─────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.getcwd()                          # folder this notebook lives in
CSV_PATH     = os.path.join(NOTEBOOK_DIR, "fake_job_postings.csv")
OUT_DIR      = os.path.join(NOTEBOOK_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("Notebook dir :", NOTEBOOK_DIR)
print("CSV expected :", CSV_PATH)
print("Output dir   :", OUT_DIR)
print("Imports done")

Notebook dir : d:\programming\fake job posting detection
CSV expected : d:\programming\fake job posting detection\fake_job_postings.csv
Output dir   : d:\programming\fake job posting detection\outputs
Imports done


In [10]:
if not os.path.exists(CSV_PATH):
    print("❌ ERROR: fake_job_postings.csv not found!")
    print("   1. Download from https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction")
    print("   2. Extract ZIP and place fake_job_postings.csv next to this notebook")
else:
    df_raw = pd.read_csv(CSV_PATH)
    print(f"Loaded: {len(df_raw):,} rows × {df_raw.shape[1]} columns")
    print(f"\nColumns: {df_raw.columns.tolist()}")
    print(f"\n── Label Distribution ──────────────────────")
    print(f"  Real jobs (0): {(df_raw['fraudulent']==0).sum():,}  ({(df_raw['fraudulent']==0).mean()*100:.1f}%)")
    print(f"  Fake jobs (1): {(df_raw['fraudulent']==1).sum():,}  ({(df_raw['fraudulent']==1).mean()*100:.1f}%)")
    df_raw.head(3)


Loaded: 17,880 rows × 18 columns

Columns: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']

── Label Distribution ──────────────────────
  Real jobs (0): 17,014  (95.2%)
  Fake jobs (1): 866  (4.8%)


In [11]:
# Show first 5 rows with key columns
cols_preview = ['job_id','title','location','salary_range',
                'employment_type','required_experience','fraudulent']
df_raw[cols_preview].head(5)


,job_id,title,location,salary_range,employment_type,required_experience,fraudulent
0,1,Marketing Intern,"US, NY, New York",NaN,Other,Internship,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",NaN,Full-time,Not Applicable,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",NaN,Full-time,Mid-Senior level,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,Full-time,Mid-Senior level,0


In [12]:
audit_rows = []
for col in df_raw.columns:
    n_null  = df_raw[col].isna().sum()
    n_empty = (df_raw[col].astype(str).str.strip() == "").sum() if df_raw[col].dtype == object else 0
    sample  = str(df_raw[col].dropna().iloc[0])[:60] if n_null < len(df_raw) else "—"
    audit_rows.append({
        "column"   : col,
        "dtype"    : str(df_raw[col].dtype),
        "nulls"    : n_null,
        "null_%"   : round(n_null / len(df_raw) * 100, 1),
        "empty_str": n_empty,
        "unique"   : df_raw[col].nunique(),
        "sample"   : sample,
    })

audit_df = pd.DataFrame(audit_rows)
print(f"Duplicate rows : {df_raw.duplicated().sum()}")
print(f"Salary present : {df_raw['salary_range'].notna().sum():,} / {len(df_raw):,} rows\n")
audit_df


Duplicate rows : 0
Salary present : 2,868 / 17,880 rows



,column,dtype,nulls,null_%,empty_str,unique,sample
0,job_id,int64,0,0.0,0,17880,1
1,title,str,0,0.0,0,11231,Marketing Intern
2,location,str,346,1.9,0,3105,"US, NY, New York"
3,department,str,11547,64.6,0,1337,Marketing
4,salary_range,str,15012,84.0,0,874,20000-28000
5,company_profile,str,3308,18.5,0,1709,"We're Food52, and we've created a groundbreaki..."
6,description,str,1,0.0,0,14801,"Food52, a fast-growing, James Beard Award-winn..."
7,requirements,str,2696,15.1,0,11967,Experience with content management systems a m...
8,benefits,str,7212,40.3,0,6204,What you will get from usThrough being part of...
9,telecommuting,int64,0,0.0,0,2,0


In [13]:
df = df_raw.copy()

# Drop exact duplicate rows
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Duplicate rows dropped : {before - len(df)}")

# Standardise column names (lowercase, underscores)
df.columns = [c.strip().lower().replace(" ","_").replace(".","_") for c in df.columns]
print(f"Column names standardised")
print(f"New column names : {df.columns.tolist()}")
print(f"Shape after dedup: {df.shape}")


Duplicate rows dropped : 0
Column names standardised
New column names : ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
Shape after dedup: (17880, 18)


In [14]:
TEXT_COLS = ["title","location","department","salary_range","company_profile",
             "description","requirements","benefits","employment_type",
             "required_experience","required_education","industry","function"]

for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# Binary flags → clean 0/1 int
for col in ["telecommuting","has_company_logo","has_questions"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Verify
null_after = df[TEXT_COLS].isnull().sum().sum()
print(f"Nulls remaining in text columns : {null_after}")
print(f"Binary flag dtypes : { {c: str(df[c].dtype) for c in ['telecommuting','has_company_logo','has_questions'] } }")
df[TEXT_COLS].isnull().sum().to_frame("nulls_remaining")


Nulls remaining in text columns : 0
Binary flag dtypes : {'telecommuting': 'int64', 'has_company_logo': 'int64', 'has_questions': 'int64'}


,nulls_remaining
title,0
location,0
department,0
salary_range,0
company_profile,0
description,0
requirements,0
benefits,0
employment_type,0
required_experience,0


In [15]:
def parse_location(s):
    parts = [p.strip() for p in str(s).split(",")]
    return (parts[0] if len(parts) > 0 else "",
            parts[1] if len(parts) > 1 else "",
            parts[2] if len(parts) > 2 else "")

locs = df["location"].apply(parse_location)
df["country"] = locs.apply(lambda x: x[0])
df["state"]   = locs.apply(lambda x: x[1])
df["city"]    = locs.apply(lambda x: x[2])

print("Top 10 countries:")
print(df["country"].value_counts().head(10).to_string())
print()
df[["location","country","state","city"]].head(8)


Top 10 countries:
country
US    10656
GB     2384
GR      940
CA      457
DE      383
        346
NZ      333
IN      276
AU      214
PH      132



,location,country,state,city
0,"US, NY, New York",US,NY,New York
1,"NZ, , Auckland",NZ,,Auckland
2,"US, IA, Wever",US,IA,Wever
3,"US, DC, Washington",US,DC,Washington
4,"US, FL, Fort Worth",US,FL,Fort Worth
5,"US, MD,",US,MD,
6,"DE, BE, Berlin",DE,BE,Berlin
7,"US, CA, San Francisco",US,CA,San Francisco


In [16]:
def parse_salary(s):
    nums = re.findall(r"\d[\d,]*", str(s).replace(",",""))
    nums = [int(n) for n in nums if n.isdigit() and 0 < int(n) < 10_000_000]
    if len(nums) >= 2: return float(min(nums)), float(max(nums))
    if len(nums) == 1: return float(nums[0]), float(nums[0])
    return np.nan, np.nan

sp = df["salary_range"].apply(parse_salary)
df["min_salary"] = sp.apply(lambda x: x[0])
df["max_salary"] = sp.apply(lambda x: x[1])
df["has_salary"] = df["min_salary"].notna().astype(int)

print(f"Rows with parseable salary : {df['has_salary'].sum():,} / {len(df):,}")
print(f"Min salary  — mean: ${df['min_salary'].mean():,.0f}   max: ${df['min_salary'].max():,.0f}")
print(f"Max salary  — mean: ${df['max_salary'].mean():,.0f}   max: ${df['max_salary'].max():,.0f}")
print()

# Compare salary presence: real vs fake
real_sal = df[df.fraudulent==0]["has_salary"].mean()*100
fake_sal = df[df.fraudulent==1]["has_salary"].mean()*100
print(f"% with salary — Real: {real_sal:.1f}%   Fake: {fake_sal:.1f}%")

df[["salary_range","min_salary","max_salary","has_salary"]].dropna(subset=["min_salary"]).head(8)


Rows with parseable salary : 2,724 / 17,880
Min salary  — mean: $65,380   max: $9,360,000
Max salary  — mean: $97,036   max: $9,360,000

% with salary — Real: 14.7%   Fake: 25.3%


,salary_range,min_salary,max_salary,has_salary
6,20000-28000,20000.0,28000.0,1
10,100000-120000,100000.0,120000.0,1
15,120000-150000,120000.0,150000.0,1
23,100000-120000,100000.0,120000.0,1
31,50000-65000,50000.0,65000.0,1
42,40000-50000,40000.0,50000.0,1
65,60-80,60.0,80.0,1
76,65000-70000,65000.0,70000.0,1


In [17]:
df["flag_title_exclamation"] = df["title"].str.count("!").gt(0).astype(int)
df["flag_title_dollar"]      = df["title"].str.contains(r"\$", regex=True).astype(int)
df["flag_title_caps_words"]  = df["title"].apply(
    lambda t: sum(1 for w in t.split() if w.isupper() and len(w) > 2))
df["flag_no_company"]        = (df["company_profile"].str.len() == 0).astype(int)
df["flag_no_requirements"]   = (df["requirements"].str.len() == 0).astype(int)
df["flag_no_benefits"]       = (df["benefits"].str.len() == 0).astype(int)
df["flag_no_description"]    = (df["description"].str.len() == 0).astype(int)

FLAG_COLS = ["flag_title_exclamation","flag_title_dollar","flag_title_caps_words",
             "flag_no_company","flag_no_requirements","flag_no_benefits",
             "flag_no_description","has_company_logo","has_salary"]

# Show rate per class
comparison = pd.DataFrame({
    "real_%": [df[df.fraudulent==0][c].mean()*100 for c in FLAG_COLS],
    "fake_%": [df[df.fraudulent==1][c].mean()*100 for c in FLAG_COLS],
}, index=FLAG_COLS).round(1)

comparison["difference"] = (comparison["fake_%"] - comparison["real_%"]).round(1)
comparison.sort_values("difference", ascending=False)


,real_%,fake_%,difference
flag_no_company,16.0,67.8,51.8
has_salary,14.7,25.3,10.6
flag_title_dollar,0.4,6.7,6.3
flag_title_caps_words,20.0,26.2,6.2
flag_no_requirements,15.0,17.8,2.8
flag_no_benefits,40.3,42.0,1.7
flag_no_description,0.0,0.2,0.2
flag_title_exclamation,1.1,0.5,-0.6
has_company_logo,81.9,32.7,-49.2


In [18]:
df["full_text"] = (
    df["title"]           + " " +
    df["company_profile"] + " " +
    df["description"]     + " " +
    df["requirements"]    + " " +
    df["benefits"]
).str.strip()

print(f"full_text created — shape: {df.shape}")
print(f"Avg chars (real): {df[df.fraudulent==0]['full_text'].str.len().mean():.0f}")
print(f"Avg chars (fake): {df[df.fraudulent==1]['full_text'].str.len().mean():.0f}")
print()
print("── Sample REAL posting ───────────────────────────────────────────")
print(df[df.fraudulent==0]["full_text"].iloc[0][:400])
print()
print("── Sample FAKE posting ───────────────────────────────────────────")
print(df[df.fraudulent==1]["full_text"].iloc[0][:400])


full_text created — shape: (17880, 32)
Avg chars (real): 2698
Avg chars (fake): 2072

── Sample REAL posting ───────────────────────────────────────────
Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly

── Sample FAKE posting ───────────────────────────────────────────
IC&E Technician Staffing &amp; Recruiting done right for the Oil &amp; Energy Industry!Represented candidates are automatically granted the following perks: Expert negotiations on your behalf, maximizing your compensation package and implimenting ongoing increases Significant signing bonus by Refined Resources (in addition to any potential signing bonuses our client companies

In [19]:
# ── Stopword list (NLTK-equivalent, built-in) ──────────────────────────────
STOPWORDS = {
    "a","an","the","and","or","but","in","on","at","to","for","of","with",
    "by","from","is","are","was","were","be","been","being","have","has",
    "had","do","does","did","will","would","could","should","may","might",
    "shall","can","it","its","itself","we","our","ours","you","your","yours",
    "he","him","his","she","her","hers","they","them","their","theirs","this",
    "that","these","those","which","who","whom","what","where","when","why",
    "how","all","each","both","few","more","most","other","some","such","no",
    "not","only","same","so","than","too","very","just","if","as","up","out",
    "about","into","through","during","before","after","above","below",
    "between","while","i","me","my","myself","get","also","within","across",
    "must","well","any","use","us","via","per","re","etc","ie","eg",
    "s","t","ll","ve","d","m",
}

# ── Lemmatizer exception dictionary (irregular + plural forms) ─────────────
LEMMA_MAP = {
    "running":"run","runs":"run","ran":"run",
    "hiring":"hire","hires":"hire","hired":"hire",
    "working":"work","worked":"work","works":"work",
    "seeking":"seek","seeks":"seek","sought":"seek",
    "managing":"manage","manages":"manage","managed":"manage",
    "building":"build","builds":"build","built":"build",
    "leading":"lead","leads":"lead","led":"lead",
    "joining":"join","joined":"join","joins":"join",
    "providing":"provide","provided":"provide","provides":"provide",
    "developing":"develop","developed":"develop","develops":"develop",
    "designing":"design","designed":"design","designs":"design",
    "looking":"look","looked":"look","looks":"look",
    "including":"include","included":"include","includes":"include",
    "required":"require","requiring":"require","requires":"require",
    "preferred":"prefer","preferring":"prefer","prefers":"prefer",
    "using":"use","used":"use","uses":"use",
    "ensuring":"ensure","ensured":"ensure","ensures":"ensure",
    "supporting":"support","supported":"support","supports":"support",
    "creating":"create","created":"create","creates":"create",
    "maintaining":"maintain","maintained":"maintain","maintains":"maintain",
    "implementing":"implement","implemented":"implement",
    "analyzing":"analyze","analysing":"analyse","analyzed":"analyze",
    "earning":"earn","earned":"earn","earns":"earn",
    "sending":"send","sends":"send","sent":"send",
    "paying":"pay","pays":"pay","paid":"pay",
    "communicating":"communicate","communicated":"communicate",
    "collaborating":"collaborate","collaborated":"collaborate",
    "experiences":"experience","responsibilities":"responsibility",
    "candidates":"candidate","skills":"skill","teams":"team",
    "positions":"position","companies":"company",
    "opportunities":"opportunity","applications":"application",
    "requirements":"requirement","salaries":"salary",
    "benefits":"benefit","roles":"role","engineers":"engineer",
    "analysts":"analyst","managers":"manager","developers":"developer",
    "designers":"designer","systems":"system","products":"product",
    "services":"service","solutions":"solution",
    "technologies":"technology","environments":"environment",
    "projects":"project","clients":"client","partners":"partner",
    "tools":"tool","platforms":"platform","frameworks":"framework",
    "databases":"database","customers":"customer","users":"user",
    "employees":"employee","tasks":"task","goals":"goal",
    "processes":"process","industries":"industry","functions":"function",
    "years":"year","months":"month","weeks":"week","days":"day",
    "abilities":"ability","qualities":"quality",
}

# ── Lightweight Porter Stemmer (fallback for words not in LEMMA_MAP) ───────
class PorterStemmer:
    RULES = [
        (r"(ational)$","ate"),(r"(tional)$","tion"),(r"(enci)$","ence"),
        (r"(anci)$","ance"),(r"(izer)$","ize"),(r"(izing)$","ize"),
        (r"(ised)$","ise"),(r"(ized)$","ize"),(r"(alism)$","al"),
        (r"(iveness)$","ive"),(r"(fulness)$","ful"),(r"(ousness)$","ous"),
        (r"(aliti)$","al"),(r"(iviti)$","ive"),(r"(biliti)$","ble"),
        (r"(ication)$","ic"),(r"(ation)$","ate"),(r"(ator)$","ate"),
        (r"(ment)$",""),(r"(ness)$",""),(r"(ful)$",""),(r"(less)$",""),
        (r"(ous)$",""),(r"(ive)$",""),(r"(ing)$",""),(r"(ed)$",""),
        (r"(ly)$",""),(r"(er)$",""),(r"(ies)$","i"),(r"(sses)$","ss"),
        (r"([^s])s$",r"\1"),
    ]
    def stem(self, word):
        if len(word) <= 3: return word
        for pat, rep in self.RULES:
            new = re.sub(pat, rep, word)
            if new != word and len(new) >= 3: return new
        return word

_stemmer = PorterStemmer()

def lemmatize(word):
    """Dictionary lookup first, Porter stemmer as fallback."""
    w = word.lower()
    return LEMMA_MAP.get(w, _stemmer.stem(w))

print(f"✅ Stopwords    : {len(STOPWORDS)} words")
print(f"✅ Lemma dict   : {len(LEMMA_MAP)} entries")
print(f"✅ PorterStemmer: ready")

# Quick demo
demo = ["running","companies","responsibilities","hiring","analyzing","developers"]
print("\nLemmatization demo:")
for w in demo:
    print(f"  {w:20} → {lemmatize(w)}")


✅ Stopwords    : 124 words
✅ Lemma dict   : 123 entries
✅ PorterStemmer: ready

Lemmatization demo:
  running              → run
  companies            → company
  responsibilities     → responsibility
  hiring               → hire
  analyzing            → analyze
  developers           → developer


In [20]:
def preprocess_text(text, trace=False):
    """
    Full NLP pipeline — 7 stages:

      Stage 1 : Strip HTML tags              <b>word</b> → word
      Stage 2 : Remove URLs & masked URLs    https://... and #URL_xxx#
      Stage 3 : Remove emails & phones       user@mail.com and #EMAIL_xxx#
      Stage 4 : Lowercase + letters only     keep [a-z ] only, collapse spaces
      Stage 5 : Tokenise                     split on whitespace
      Stage 6 : Remove stopwords             drop 'the','and','is' etc + len<=2
      Stage 7 : Lemmatise                    dict lookup → Porter stemmer fallback

    Args:
        text  : raw string from full_text column
        trace : if True, returns every intermediate stage (good for debugging)

    Returns dict with keys: tokens, tokens_no_sw, lemmatized, final_text
    (+ intermediate stages when trace=True)
    """
    if not isinstance(text, str) or not text.strip():
        out = {"tokens":[], "tokens_no_sw":[], "lemmatized":[], "final_text":""}
        if trace:
            out.update({"raw":"","no_html":"","no_urls":"","no_emails":"","clean_text":""})
        return out

    # Stage 1 — strip HTML
    no_html   = re.sub(r"<[^>]+>", " ", text)

    # Stage 2 — remove URLs (real + EMSCAD masked #URL_xxx#)
    no_urls   = re.sub(r"https?://\S+|www\.\S+|#URL_[A-Za-z0-9_]+#", " ", no_html)

    # Stage 3 — remove emails / phones (real + EMSCAD masked)
    no_emails = re.sub(
        r"\S+@\S+|#EMAIL_[A-Za-z0-9_]+#|#PHONE_[A-Za-z0-9_]+#", " ", no_urls)

    # Stage 4 — lowercase + letters only + collapse whitespace
    clean     = re.sub(r"[^a-zA-Z\s]", " ", no_emails)
    clean     = re.sub(r"\s+", " ", clean).strip().lower()

    # Stage 5 — tokenise
    tokens    = clean.split()

    # Stage 6 — remove stopwords + very short tokens
    no_sw     = [t for t in tokens if t not in STOPWORDS and len(t) > 2]

    # Stage 7 — lemmatise
    lemmas    = [lemmatize(t) for t in no_sw]
    final     = " ".join(lemmas)

    if trace:
        return {
            "raw"          : text[:500],
            "no_html"      : no_html[:500],
            "no_urls"      : no_urls[:500],
            "no_emails"    : no_emails[:500],
            "clean_text"   : clean[:500],
            "tokens"       : tokens[:30],
            "tokens_no_sw" : no_sw[:30],
            "lemmatized"   : lemmas[:30],
            "final_text"   : final[:500],
        }

    return {"tokens":tokens, "tokens_no_sw":no_sw,
            "lemmatized":lemmas, "final_text":final}

print("✅ preprocess_text() defined")


✅ preprocess_text() defined


In [21]:
# Show full trace for one REAL and one FAKE posting
for label, subset in [("REAL", df[df.fraudulent==0]), ("FAKE", df[df.fraudulent==1])]:
    sample = subset.sample(1, random_state=42).iloc[0]
    trace  = preprocess_text(sample["full_text"], trace=True)
    print(f"{'='*65}")
    print(f"  {label} — {sample['title']}")
    print(f"{'='*65}")
    for stage in ["raw","no_html","no_urls","no_emails","clean_text",
                  "tokens","tokens_no_sw","lemmatized"]:
        val = trace[stage]
        if isinstance(val, list):
            print(f"  {stage:<15}: {val[:12]}")
        else:
            print(f"  {stage:<15}: {str(val)[:120]}")
    print(f"  {'final_text':<15}: {trace['final_text'][:150]}")
    print()


  REAL — SEM Coordinator
  raw            : SEM Coordinator #URL_c379aa631173ed5b7c345ab3f500a9a053e509138ca70e52c1088e5a784dc8d7# is a modern online travel agency 
  no_html        : SEM Coordinator #URL_c379aa631173ed5b7c345ab3f500a9a053e509138ca70e52c1088e5a784dc8d7# is a modern online travel agency 
  no_urls        : SEM Coordinator   is a modern online travel agency that was founded in Athens in 2005. Today, our company is one of the 
  no_emails      : SEM Coordinator   is a modern online travel agency that was founded in Athens in 2005. Today, our company is one of the 
  clean_text     : sem coordinator is a modern online travel agency that was founded in athens in today our company is one of the leading o
  tokens         : ['sem', 'coordinator', 'is', 'a', 'modern', 'online', 'travel', 'agency', 'that', 'was', 'founded', 'in']
  tokens_no_sw   : ['sem', 'coordinator', 'modern', 'online', 'travel', 'agency', 'founded', 'athens', 'today', 'company', 'one', 'leading']
  lemmati

In [22]:
print(f"Processing {len(df):,} rows...")

nlp_rows = []
for i, row in df.iterrows():
    r = preprocess_text(row["full_text"])
    nlp_rows.append({
        "job_id"                 : row.get("job_id", i),
        "fraudulent"             : row["fraudulent"],
        "title"                  : row["title"],
        "country"                : row["country"],
        "state"                  : row["state"],
        "employment_type"        : row["employment_type"],
        "required_experience"    : row["required_experience"],
        "required_education"     : row["required_education"],
        "industry"               : row["industry"],
        "has_company_logo"       : row["has_company_logo"],
        "telecommuting"          : row["telecommuting"],
        "has_questions"          : row["has_questions"],
        "has_salary"             : row["has_salary"],
        "min_salary"             : row["min_salary"],
        "max_salary"             : row["max_salary"],
        "flag_title_exclamation" : row["flag_title_exclamation"],
        "flag_title_dollar"      : row["flag_title_dollar"],
        "flag_title_caps_words"  : row["flag_title_caps_words"],
        "flag_no_company"        : row["flag_no_company"],
        "flag_no_requirements"   : row["flag_no_requirements"],
        "flag_no_benefits"       : row["flag_no_benefits"],
        "flag_no_description"    : row["flag_no_description"],
        "raw_full_text"          : row["full_text"][:600],
        "final_text"             : r["final_text"],
        "tokens"                 : json.dumps(r["tokens"][:60]),
        "tokens_no_sw"           : json.dumps(r["tokens_no_sw"][:60]),
        "lemmatized_tokens"      : json.dumps(r["lemmatized"][:60]),
        "token_count_raw"        : len(r["tokens"]),
        "token_count_clean"      : len(r["lemmatized"]),
    })
    if (i + 1) % 3000 == 0:
        print(f"  ... {i+1:,} / {len(df):,} done")

df_nlp    = pd.DataFrame(nlp_rows)
avg_raw   = df_nlp["token_count_raw"].mean()
avg_clean = df_nlp["token_count_clean"].mean()
reduction = (1 - avg_clean / avg_raw) * 100 if avg_raw > 0 else 0

print(f"\n✅ Done — {len(df_nlp):,} rows processed")
print(f"   Avg raw tokens   : {avg_raw:.1f}")
print(f"   Avg clean tokens : {avg_clean:.1f}")
print(f"   Token reduction  : {reduction:.1f}%  (stopwords + noise removed)")
df_nlp[["job_id","fraudulent","title","token_count_raw","token_count_clean","final_text"]].head(5)


Processing 17,880 rows...
  ... 3,000 / 17,880 done
  ... 6,000 / 17,880 done
  ... 9,000 / 17,880 done
  ... 12,000 / 17,880 done
  ... 15,000 / 17,880 done

✅ Done — 17,880 rows processed
   Avg raw tokens   : 386.9
   Avg clean tokens : 238.7
   Token reduction  : 38.3%  (stopwords + noise removed)


,job_id,fraudulent,title,token_count_raw,token_count_clean,final_text
0,1,0,Marketing Intern,403,251,market intern food create groundbreak award wi...
1,2,0,Customer Service - Cloud Video Production,873,519,custom service cloud video production second w...
2,3,0,Commissioning Machinery Assistant (CMA),384,232,commission machinery assistant cma valor servi...
3,4,0,Account Executive - Washington DC,732,481,account execut washington passion improv quali...
4,5,0,Bill Review Manager,489,343,bill review manag spotsource solution llc glob...


In [23]:
real_tok, fake_tok = [], []
for _, row in df_nlp.iterrows():
    toks = json.loads(row["lemmatized_tokens"])
    (fake_tok if row["fraudulent"] else real_tok).extend(toks)

freq_real = Counter(real_tok)
freq_fake = Counter(fake_tok)

print(f"Vocabulary (real postings): {len(freq_real):,} unique lemmas")
print(f"Vocabulary (fake postings): {len(freq_fake):,} unique lemmas")

top_n = 20
real_top = pd.DataFrame(freq_real.most_common(top_n), columns=["lemma","count_real"])
fake_top = pd.DataFrame(freq_fake.most_common(top_n), columns=["lemma","count_fake"])

print("\n── Top 20 REAL lemmas ───────────────────────────────────────────────")
print(real_top.to_string(index=False))
print("\n── Top 20 FAKE lemmas ───────────────────────────────────────────────")
print(fake_top.to_string(index=False))


Vocabulary (real postings): 19,042 unique lemmas
Vocabulary (fake postings): 3,874 unique lemmas

── Top 20 REAL lemmas ───────────────────────────────────────────────
     lemma  count_real
   company       10518
   service        9964
      work        9486
      team        8064
       job        7545
   develop        7512
   provide        6293
      busi        5805
  solution        5692
experience        5418
    client        5367
technology        5228
      help        5211
    people        5139
    manage        5041
    market        4876
   product        4772
       new        4669
    design        4282
      lead        4225

── Top 20 FAKE lemmas ───────────────────────────────────────────────
     lemma  count_fake
   service         649
      work         601
  position         499
      time         416
experience         405
   company         358
       amp         351
       job         331
      data         320
      busi         307
      home         296
  

In [24]:
FLAG_COLS = ["flag_no_company","flag_no_requirements","flag_no_benefits",
             "flag_no_description","flag_title_exclamation","flag_title_dollar",
             "flag_title_caps_words","has_company_logo","has_salary",
             "telecommuting","has_questions"]

eda = pd.DataFrame({
    "real_%": [df_nlp[df_nlp.fraudulent==0][c].mean()*100 for c in FLAG_COLS],
    "fake_%": [df_nlp[df_nlp.fraudulent==1][c].mean()*100 for c in FLAG_COLS],
}, index=FLAG_COLS).round(2)

eda["difference_(fake-real)"] = (eda["fake_%"] - eda["real_%"]).round(2)
eda.sort_values("difference_(fake-real)", ascending=False)


,real_%,fake_%,difference_(fake-real)
flag_no_company,15.99,67.78,51.79
has_salary,14.72,25.29,10.57
flag_title_dollar,0.42,6.70,6.28
flag_title_caps_words,20.04,26.21,6.17
telecommuting,4.13,7.39,3.26
flag_no_requirements,14.95,17.78,2.83
flag_no_benefits,40.30,42.03,1.73
flag_no_description,0.00,0.23,0.23
flag_title_exclamation,1.13,0.46,-0.67
has_questions,50.21,28.87,-21.34


In [25]:
print("── Employment Type ─────────────────────────────────────────────────")
print(df["employment_type"].value_counts().to_string())

print("\n── Required Education ──────────────────────────────────────────────")
print(df["required_education"].value_counts().to_string())

print("\n── Required Experience ─────────────────────────────────────────────")
print(df["required_experience"].value_counts().to_string())

print("\n── Top Industries ──────────────────────────────────────────────────")
print(df["industry"].value_counts().head(15).to_string())


── Employment Type ─────────────────────────────────────────────────
employment_type
Full-time    11620
              3471
Contract      1524
Part-time      797
Temporary      241
Other          227

── Required Education ──────────────────────────────────────────────
required_education
                                     8105
Bachelor's Degree                    5145
High School or equivalent            2080
Unspecified                          1397
Master's Degree                       416
Associate Degree                      274
Certification                         170
Some College Coursework Completed     102
Professional                           74
Vocational                             49
Some High School Coursework            27
Doctorate                              26
Vocational - HS Diploma                 9
Vocational - Degree                     6

── Required Experience ─────────────────────────────────────────────
required_experience
                    7050
Mid-Senio

In [26]:
# 1. Cleaned structured dataset
df.to_csv(f"{OUT_DIR}/cleaned_jobs.csv", index=False)
print(f"✅ cleaned_jobs.csv           {len(df):,} rows  {df.shape[1]} cols")

# 2. NLP-processed dataset (use this for modelling)
df_nlp.to_csv(f"{OUT_DIR}/nlp_processed_jobs.csv", index=False)
print(f"✅ nlp_processed_jobs.csv     {len(df_nlp):,} rows  {df_nlp.shape[1]} cols  ← Phase 2 input")

# 3. Vocabulary JSONs
with open(f"{OUT_DIR}/vocabulary_real.json","w") as f:
    json.dump({w:c for w,c in freq_real.most_common(1000)}, f, indent=2)
with open(f"{OUT_DIR}/vocabulary_fake.json","w") as f:
    json.dump({w:c for w,c in freq_fake.most_common(1000)}, f, indent=2)
print(f"✅ vocabulary_real/fake.json  top 1,000 lemmas each")

# 4. Pipeline trace (5 real + 5 fake with every NLP stage)
trace_idx = (list(df[df.fraudulent==0].sample(5, random_state=42).index) +
             list(df[df.fraudulent==1].sample(5, random_state=42).index))
traces = []
for idx in trace_idx:
    row = df.iloc[idx]
    traces.append({
        "job_id"   : int(row.get("job_id", idx)),
        "label"    : "FAKE" if row["fraudulent"] else "REAL",
        "title"    : row["title"],
        "pipeline" : preprocess_text(row["full_text"], trace=True),
    })
with open(f"{OUT_DIR}/pipeline_trace.json","w") as f:
    json.dump(traces, f, indent=2)
print(f"✅ pipeline_trace.json        10 step-by-step NLP examples")

print(f"\n📁 All files saved to: {OUT_DIR}/")


✅ cleaned_jobs.csv           17,880 rows  32 cols
✅ nlp_processed_jobs.csv     17,880 rows  29 cols  ← Phase 2 input
✅ vocabulary_real/fake.json  top 1,000 lemmas each
✅ pipeline_trace.json        10 step-by-step NLP examples

📁 All files saved to: d:\programming\fake job posting detection\outputs/


In [27]:
print("=" * 60)
print("  PHASE 1 COMPLETE")
print("=" * 60)
print(f"  Dataset rows     : {len(df_nlp):,}")
print(f"  Real postings    : {(df_nlp.fraudulent==0).sum():,}  ({(df_nlp.fraudulent==0).mean()*100:.1f}%)")
print(f"  Fake postings    : {(df_nlp.fraudulent==1).sum():,}  ({(df_nlp.fraudulent==1).mean()*100:.1f}%)")
print(f"  Vocabulary (real): {len(freq_real):,} unique lemmas")
print(f"  Vocabulary (fake): {len(freq_fake):,} unique lemmas")
print(f"  Token reduction  : {reduction:.1f}%")
print()
print("  Output files:")
for f in os.listdir(OUT_DIR):
    path = os.path.join(OUT_DIR, f)
    size = os.path.getsize(path) / 1024
    print(f"    {f:<35} {size:>8.1f} KB")
print()
print("  Phase 2 — Feature Extraction:")
print("    • TF-IDF on final_text          sklearn TfidfVectorizer")
print("    • Word2Vec / BERT embeddings    gensim / transformers")
print("    • Salary z-score anomaly        min_salary / max_salary")
print("    • Train classifier              Logistic Regression / Random Forest / BERT")


  PHASE 1 COMPLETE
  Dataset rows     : 17,880
  Real postings    : 17,014  (95.2%)
  Fake postings    : 866  (4.8%)
  Vocabulary (real): 19,042 unique lemmas
  Vocabulary (fake): 3,874 unique lemmas
  Token reduction  : 38.3%

  Output files:
    cleaned_jobs.csv                     96570.3 KB
    nlp_processed_jobs.csv               82062.4 KB
    pipeline_trace.json                     51.4 KB
    vocabulary_fake.json                    17.1 KB
    vocabulary_real.json                    18.1 KB

  Phase 2 — Feature Extraction:
    • TF-IDF on final_text          sklearn TfidfVectorizer
    • Word2Vec / BERT embeddings    gensim / transformers
    • Salary z-score anomaly        min_salary / max_salary
    • Train classifier              Logistic Regression / Random Forest / BERT
